<a href="https://colab.research.google.com/github/kostismatz/GKS_ML_Project/blob/main/2BiRNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
print(torch.cuda.is_available())
import torch
print(torch.__version__)



# -*- coding: utf-8 -*-
"""

A RNN classifier applied to AG_NEWS dataset

Download dataset:
https://www.kaggle.com/datasets/amananandrai/ag-news-classification-dataset

"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import time
from torch.utils.data import DataLoader
from torch import nn
from torch.nn import functional as F
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from collections import Counter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

MAX_WORDS = 25
EPOCHS = 15
LEARNING_RATE = 1e-3
BATCH_SIZE = 1024
EMBEDDING_DIM = 100
HIDDEN_DIM = 64

train_data = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')

######################################################################
# Data processing
# -----------------------------


def tokenizer(text):
    return text.lower().split()

# All texts are truncated and padded to MAX_WORDS tokens
def collate_batch(batch):
    Y, X = zip(*batch)

    Y = torch.tensor(Y, dtype=torch.long) - 1

    X_processed = []

    for text in X:
        tokens = tokenizer(text)

        indices = []
        for token in tokens:
            if token in vocab:
                indices.append(vocab[token])
            else:
                indices.append(vocab["<UNK>"])

        if len(indices) < MAX_WORDS:
            indices += [vocab["<PAD>"]] * (MAX_WORDS - len(indices))
        else:
            indices = indices[:MAX_WORDS]

        X_processed.append(indices)

    X_tensor = torch.tensor(X_processed, dtype=torch.long)

    return X_tensor.to(device), Y.to(device)

train_dataset = [(label,train_data['Title'][i] + ' ' + train_data['Description'][i]) for i,label in enumerate(train_data['Class Index'])]
test_dataset = [(label,test_data['Title'][i] + ' ' + test_data['Description'][i]) for i,label in enumerate(test_data['Class Index'])]

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                              shuffle=True, collate_fn=collate_batch)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                              shuffle=False, collate_fn=collate_batch)

target_classes = ["World", "Sports", "Business", "Sci/Tech"]

def build_vocabulary(dataset, min_freq=10):
    counter = Counter()

    for _, text in dataset:
        tokens = tokenizer(text)
        counter.update(tokens)

    vocab_dict = {
        "<PAD>": 0,
        "<UNK>": 1
    }

    idx = 2
    for token, freq in counter.items():
        if freq >= min_freq:
            vocab_dict[token] = idx
            idx += 1

    return vocab_dict

vocab = build_vocabulary(train_dataset, min_freq=10)


class model(nn.Module):
    def __init__(self, input_dim, embedding_dim, hidden_dim, output_dim):
        super(model, self).__init__()

        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            bidirectional=True
        )

        self.embedding_layer = nn.Embedding(input_dim, embedding_dim)

        self.linear = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, X_batch):
        embeddings = self.embedding_layer(X_batch)

        output, hidden = self.rnn(embeddings)

        # 🔥 σωστό BiRNN pooling
        out = torch.cat((hidden[-2], hidden[-1]), dim=1)

        return self.linear(out)


 # Initiate an instance of the model
# ---------------------------------


classifier = model(len(vocab), EMBEDDING_DIM, HIDDEN_DIM, len(target_classes)).to(device)
# Define loss function and opimization algorithm
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam([param for param in classifier.parameters() if param.requires_grad == True],lr=LEARNING_RATE)

# Count model parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print('\nModel:')
print(classifier)
print('Total parameters: ',count_parameters(classifier))
print('\n\n')

######################################################################



# Define functions to train and evaluate the model
# ------------------------------------------------


def EvaluateModel(model, loss_fn, val_loader):
    model.eval()
    with torch.no_grad():

        Y_actual, Y_preds, losses = [], [], []

        for X, Y in val_loader:

            # 🔥 MOVE TO GPU
            X = X.to(device)
            Y = Y.to(device)

            preds = model(X)
            loss = loss_fn(preds, Y)

            losses.append(loss.item())

            Y_actual.append(Y)
            Y_preds.append(preds.argmax(dim=-1))

        Y_actual = torch.cat(Y_actual)
        Y_preds = torch.cat(Y_preds)

    return torch.tensor(losses).mean(), Y_actual.cpu().numpy(), Y_preds.cpu().numpy()


def TrainModel(model, loss_fn, optimizer, train_loader, epochs):
    epoch_times = []

    for i in range(1, epochs+1):
        model.train()
        print('Epoch:', i)

        losses = []

        start_time = time.time()

        for X, Y in tqdm(train_loader):

            # 🔥 MOVE TO GPU
            X = X.to(device)
            Y = Y.to(device)

            Y_preds = model(X)
            loss = loss_fn(Y_preds, Y)

            losses.append(loss.item())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        end_time = time.time()

        epoch_time = end_time - start_time
        epoch_times.append(epoch_time)

        print("Train Loss : {:.3f}".format(torch.tensor(losses).mean()))
        print("Epoch Time (sec): {:.2f}".format(epoch_time))

    print("\nAverage Epoch Time:", sum(epoch_times)/len(epoch_times))

import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

TrainModel(classifier, loss_fn, optimizer, train_loader, EPOCHS)


# Evaluate the model with test dataset
# ------------------------------------


_, Y_actual, Y_preds = EvaluateModel(classifier, loss_fn, test_loader)

print("\nTest Accuracy : {:.3f}".format(accuracy_score(Y_actual, Y_preds)))
print("\nClassification Report : ")
print(classification_report(Y_actual, Y_preds, target_names=target_classes))
print("\nConfusion Matrix : ")
print(confusion_matrix(Y_actual, Y_preds))


# =========================
# 🔁 3 RUN EXPERIMENT
# =========================

from sklearn.metrics import accuracy_score
import numpy as np

accuracies = []

for run in range(3):
    print(f"\n===== RUN {run+1} =====")

    # νέο model κάθε φορά
    classifier = model(len(vocab), EMBEDDING_DIM, HIDDEN_DIM, len(target_classes)).to(device)

    optimizer = torch.optim.Adam(classifier.parameters(), lr=LEARNING_RATE)

    # training
    TrainModel(classifier, loss_fn, optimizer, train_loader, EPOCHS)

    # evaluation
    _, y_true, y_pred = EvaluateModel(classifier, loss_fn, test_loader)

    acc = accuracy_score(y_true, y_pred)
    accuracies.append(acc)

    print("Test Accuracy:", acc)


    print("\n===== FINAL RESULTS =====")

mean_acc = np.mean(accuracies)
std_acc = np.std(accuracies)

print("Accuracies:", accuracies)
print("Mean Accuracy:", mean_acc)
print("Std:", std_acc)





True
2.10.0+cu128
cuda

Model:
model(
  (rnn): RNN(100, 64, num_layers=2, batch_first=True, bidirectional=True)
  (embedding_layer): Embedding(25099, 100)
  (linear): Linear(in_features=128, out_features=4, bias=True)
)
Total parameters:  2556496



Epoch: 1


100%|██████████| 118/118 [00:03<00:00, 35.38it/s]


Train Loss : 1.033
Epoch Time (sec): 3.34
Epoch: 2


100%|██████████| 118/118 [00:02<00:00, 47.86it/s]


Train Loss : 0.564
Epoch Time (sec): 2.47
Epoch: 3


...

Train Loss : 0.111
Epoch Time (sec): 2.02
Epoch: 14


100%|██████████| 118/118 [00:01<00:00, 59.48it/s]


Train Loss : 0.094
Epoch Time (sec): 1.99
Epoch: 15


100%|██████████| 118/118 [00:01<00:00, 59.53it/s]


Train Loss : 0.081
Epoch Time (sec): 1.99

Average Epoch Time: 2.169016981124878
Test Accuracy: 0.883421052631579

===== FINAL RESULTS =====
Accuracies: [0.8759210526315789, 0.8786842105263157, 0.883421052631579]
Mean Accuracy: 0.8793421052631579
Std: 0.00309700060419472


# New Section